# GNSS Velocity from Doppler

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/DopplerVelocityExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview
Besides the pseudorange and the carrier phase, a GNSS receiver measures the **Doppler shift** of every signal it tracks: the rate at which the relative motion of satellite and receiver stretches the carrier. One Doppler observation is a *range-rate* measurement along the line of sight, so a handful of satellites determines all three components of the receiver velocity -- and, as a by-product, the drift of the receiver clock.

`DopplerFactor` puts that measurement into a factor graph. Its keys are the receiver velocity at epoch $k$ and the receiver clock bias at epochs $k-1$ and $k$, because the clock *drift* is not a separate state but the difference of two adjacent clock biases:

$$e^{T}(v_s - v_r) + c\left(\frac{b_k - b_{k-1}}{\Delta t} - \dot b_s\right) + \text{Sagnac rate} \; - \; (-\lambda D).$$

This notebook runs the factor on real 1 Hz data from the **same open-sky Septentrio mosaic-X5 receiver as Part 2 of [`RtkAndPppExample.ipynb`](RtkAndPppExample.ipynb)**. The antenna sits on a surveyed static marker, so its true velocity is exactly zero and every metre per second we estimate is error. The second half of the notebook uses the same data to exercise `DopplerFactorArm`, the lever-arm variant for an antenna offset from a rotating body.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

In [ ]:
try:
    import google.colab  # noqa: F401
    %pip install --quiet numpy plotly gtsam-develop \
        "git+https://github.com/inuex35/cssrlib-numba.git@gtsam-gnss-frontend"
    !wget -q https://raw.githubusercontent.com/borglab/gtsam/develop/python/gtsam/examples/gnss_frontend.py
except ImportError:
    pass

In [ ]:
import os
import urllib.request

import numpy as np
import plotly.graph_objects as go

import gtsam
from gtsam import symbol
import gnss_frontend as gnss
from cssrlib.gnss import ecef2enu, rCST, xyz2enu

# The observation and broadcast navigation files of the open-sky dataset; the
# CLAS corrections of the PPP example are not needed for Doppler.
DATADIR = os.environ.get("GNSS_DATA", "gnss_data")
_BASE = "https://raw.githubusercontent.com/hirokawa/cssrlib-data/main/data"
for name in ("233h_rnx.obs", "233h_rnx.nav"):
    dst = os.path.join(DATADIR, "doy2025-233", name)
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        print("downloading", name, "...")
        urllib.request.urlretrieve(f"{_BASE}/doy2025-233/{name}", dst)
print("data ready in", DATADIR)

## The measurements

`gnss_frontend.load_doppler` decodes the RINEX, computes the satellite states from the broadcast ephemeris and keeps the L1 Doppler of the healthy GPS/Galileo/QZSS satellites above a 15° elevation mask. Each record carries what the factor needs as constants: the satellite ECEF position and velocity, its clock offset, the line of sight and the carrier wavelength.

In [ ]:
EPOCHS = 300
# One extra epoch: the first one only anchors the clock-bias chain.
data = gnss.load_doppler(DATADIR, n_epochs=EPOCHS + 1)
print(f"{len(data.frames)} epochs at 1 Hz, "
      f"{np.mean([len(f.sats) for f in data.frames]):.1f} satellites per epoch")
print(f"surveyed marker (true velocity = 0): {data.xyz_ref}")

## The factor graph

One velocity node and one clock-bias node per epoch. Each Doppler observation becomes a `DopplerFactor` on `[v_k, b_{k-1}, b_k]`, weighted by $1/\sin(\text{elevation})$. Only differences of the clock biases are observable, so the chain has one gauge freedom, pinned with a prior on the first bias. The graph is linear in all its states, so a single Levenberg-Marquardt iteration solves it.

In [ ]:
V = lambda k: symbol('v', k)   # receiver ECEF velocity at epoch k
B = lambda k: symbol('b', k)   # receiver clock bias at epoch k
SIGMA = 0.05                   # zenith range-rate sigma [m/s]

graph, initial = gtsam.NonlinearFactorGraph(), gtsam.Values()
initial.insert(B(0), 0.0)
graph.addPriorDouble(B(0), 0.0, gtsam.noiseModel.Isotropic.Sigma(1, 1e-9))

factors = []
for k in range(1, len(data.frames)):
    initial.insert(V(k), gtsam.Point3(0, 0, 0))
    initial.insert(B(k), 0.0)
    for o in gnss.doppler_observations(data.frames[k - 1], data.frames[k], data):
        factor = gtsam.DopplerFactor(
            V(k), B(k - 1), B(k), *o.meas_args, *o.epoch_args,
            gtsam.noiseModel.Isotropic.Sigma(1, SIGMA * o.w))
        graph.add(factor)
        factors.append((k, o, factor))

result = gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()
print(f"{len(factors)} Doppler factors, {result.size()} states")

## How fast is a station that does not move?

The antenna is static, so the estimated velocity *is* the error. We express it in the local East/North/Up frame of the marker.

In [ ]:
velocity = np.array([result.atPoint3(V(k)) for k in range(1, len(data.frames))])
enu = np.array([ecef2enu(data.pos_ref, v) for v in velocity])

print("velocity error (truth = 0):")
print("  bias  E/N/U = %+.4f %+.4f %+.4f m/s" % tuple(enu.mean(axis=0)))
print("  RMS   E/N/U =  %.4f  %.4f  %.4f m/s" % tuple(np.sqrt((enu ** 2).mean(axis=0))))
print("  horizontal RMS %.4f m/s, 3D RMS %.4f m/s, worst epoch %.4f m/s"
      % (np.sqrt((enu[:, :2] ** 2).sum(axis=1).mean()),
         np.sqrt((enu ** 2).sum(axis=1).mean()),
         np.linalg.norm(enu, axis=1).max()))

In [ ]:
fig = go.Figure()
for i, (name, color) in enumerate(zip(("East", "North", "Up"),
                                      ("#1f77b4", "#2ca02c", "#d62728"))):
    fig.add_scatter(y=enu[:, i], name=name, line=dict(width=1, color=color))
fig.update_layout(
    title="Velocity error of a static antenna, estimated from Doppler alone",
    xaxis_title="epoch [s]", yaxis_title="velocity error [m/s]",
    height=380, margin=dict(l=60, r=20, t=50, b=50))
fig.show()

A few centimetres per second, with no bias to speak of: the range-rate model -- line of sight, satellite velocity, satellite clock drift and the Earth-rotation (Sagnac) rate -- is consistent with the measurements at the level the receiver can measure them.

The other state the graph estimates is the receiver clock. Its drift is the difference of adjacent bias states, and the post-fit residuals show the elevation dependence the weighting assumed.

In [ ]:
drift = rCST.CLIGHT * np.diff([result.atDouble(B(k))
                               for k in range(len(data.frames))])
print("receiver clock drift: mean %+.4f m/s, std %.4f m/s"
      % (drift.mean(), drift.std()))

residual = np.array([f.evaluateError(result.atPoint3(V(k)),
                                     result.atDouble(B(k - 1)),
                                     result.atDouble(B(k)))[0]
                     for k, o, f in factors])
elevation = np.rad2deg([o.el for _, o, _ in factors])
print("range-rate residual: RMS %.4f m/s over %d observations"
      % (np.sqrt((residual ** 2).mean()), len(residual)))

fig = go.Figure()
fig.add_scatter(x=elevation, y=residual, mode="markers",
                marker=dict(size=3, opacity=0.35, color="#1f77b4"), name="residual")
fig.add_scatter(x=np.arange(15, 91), y=SIGMA / np.sin(np.deg2rad(np.arange(15, 91))),
                line=dict(color="#d62728", dash="dash"), name="assumed 1-sigma")
fig.add_scatter(x=np.arange(15, 91), y=-SIGMA / np.sin(np.deg2rad(np.arange(15, 91))),
                line=dict(color="#d62728", dash="dash"), showlegend=False)
fig.update_layout(title="Post-fit range-rate residuals",
                  xaxis_title="satellite elevation [deg]",
                  yaxis_title="residual [m/s]",
                  height=380, margin=dict(l=60, r=20, t=50, b=50))
fig.show()

## Lever arm: `DopplerFactorArm`

On a vehicle the antenna is bolted somewhere other than the body origin, so when the body rotates the antenna moves even if the body frame does not:

$$v_\text{antenna} = v_\text{body} + R_\text{body}^\text{ECEF}\,(\omega \times \ell).$$

`DopplerFactorArm` adds a `Pose3` key for the body attitude and takes the body-frame lever arm $\ell$ and angular rate $\omega$ as constants. With $\omega = 0$ it must reduce exactly to `DopplerFactor`, which is the first thing to check on real data.

In [ ]:
P0 = symbol('p', 0)
# Body axes aligned with the local ENU frame of the marker.
ecef_R_body = gtsam.Rot3(xyz2enu(data.pos_ref).T)
pose = gtsam.Pose3(ecef_R_body, gtsam.Point3(*data.xyz_ref))
LEVER = np.array([0.5, 0.0, 1.0])     # antenna offset in the body frame [m]


def solve_with_arm(omega, inject=False):
    """Optimize the same data with DopplerFactorArm for a given body rate.

    With ``inject`` the measured Doppler is modified by the range rate the
    lever arm would produce, i.e. we pretend the body spun while the antenna
    physically stayed put -- the factor should then return a zero body velocity.
    """
    lever_velocity = ecef_R_body.matrix() @ np.cross(omega, LEVER)
    graph, initial = gtsam.NonlinearFactorGraph(), gtsam.Values()
    initial.insert(P0, pose)
    initial.insert(B(0), 0.0)
    graph.add(gtsam.PriorFactorPose3(
        P0, pose, gtsam.noiseModel.Isotropic.Sigma(6, 1e-6)))
    graph.addPriorDouble(B(0), 0.0, gtsam.noiseModel.Isotropic.Sigma(1, 1e-9))
    for k in range(1, len(data.frames)):
        initial.insert(V(k), gtsam.Point3(0, 0, 0))
        initial.insert(B(k), 0.0)
        for o in gnss.doppler_observations(data.frames[k - 1], data.frames[k], data):
            doppler, *rest = o.meas_args
            if inject:
                doppler += (o.los @ lever_velocity) / o.lam
            graph.add(gtsam.DopplerFactorArm(
                P0, V(k), B(k - 1), B(k), doppler, *rest,
                gtsam.Point3(*LEVER), gtsam.Point3(*omega), *o.epoch_args,
                gtsam.noiseModel.Isotropic.Sigma(1, SIGMA * o.w)))
    res = gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()
    body = np.array([res.atPoint3(V(k)) for k in range(1, len(data.frames))])
    return np.array([ecef2enu(data.pos_ref, v) for v in body])


still = solve_with_arm(np.zeros(3))
print("max |DopplerFactorArm(omega=0) - DopplerFactor| = %.2e m/s"
      % np.abs(still - enu).max())

Now spin the body at 0.5 rad/s about its up axis. The lever arm swings the antenna through $\omega \times \ell = (0, 0.25, 0)$ m/s in the body frame, but the *real* antenna in this dataset never moved. So the factor, told to expect that antenna motion, must attribute it to a body that translates the other way -- and it does, up to the same few-millimetre-per-second offset the static solution already carried.

Feeding the factor a Doppler measurement that also carries the swing (`inject=True`) restores a body at rest: the lever-arm term and the injected motion cancel, on real satellite geometry.

In [ ]:
OMEGA = np.array([0.0, 0.0, 0.5])       # body rate [rad/s], spin about up
expected = np.cross(OMEGA, LEVER)       # antenna motion in the body frame [m/s]

spinning = solve_with_arm(OMEGA)
corrected = solve_with_arm(OMEGA, inject=True)

print("expected antenna swing  E/N/U = %+.4f %+.4f %+.4f m/s" % tuple(expected))
for name, v in (("omega = 0", still), ("spinning body", spinning),
                ("spinning body, Doppler carries the swing", corrected)):
    print("%-42s mean E/N/U = %+.4f %+.4f %+.4f m/s" % (name, *v.mean(axis=0)))

## Conclusion

Doppler alone, with no position states and no atmosphere model, pins the velocity of a static geodetic antenna to a few centimetres per second and its clock drift to a few tenths of a nanosecond per second. In a real navigation graph these factors sit next to `PseudorangeFactor` or the double-difference factors, sharing the same clock-bias chain, and give the velocity observability that code measurements alone provide only through differencing.

## Sources
- [DopplerFactor.h](https://github.com/borglab/gtsam/blob/develop/gtsam/navigation/DopplerFactor.h) / [DopplerFactor.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/navigation/DopplerFactor.cpp)
- [DopplerFactor.ipynb](../../../gtsam/navigation/doc/DopplerFactor.ipynb) -- the factor's reference documentation
- [RtkAndPppExample.ipynb](RtkAndPppExample.ipynb) -- same receiver and dataset, carrier-phase positioning
- Front-end helper: `gnss_frontend.py`; open-sky [dataset](https://github.com/hirokawa/cssrlib-data)